# 07 — Train (RunPod / GPU)

Full training pipeline. One execution = one experiment run stored as a self-contained HF model repo.

**What this notebook does:**
1. Pulls curriculum JSONL files from a source HF dataset repo
2. Resumes automatically if the pod is interrupted
3. Trains with `train_word_aware` — saves `chck_1M`…`chck_10M` as HF git revisions (competition format) and `phase_XX_epoch_YY` folders for analysis
4. Pushes `run_config.json` and `training_history.json` to the model repo
5. Plots and saves training curves

**Resume:** re-run all cells. Cell 5 finds the latest checkpoint on Hub and picks up from there.

**Eval:** run notebook 08 after this one finishes (same or separate pod).

In [ ]:
# ── Cell 1: RunPod setup (run once per session) ──
# Uncomment and run this block on a fresh RunPod session.

# !git clone https://github.com/flakoash/influece_driven_curriculum_sorter.git /workspace/babylm
# %cd /workspace/babylm
# !pip install -e '.[dev]' --quiet

print('Setup complete.')

In [ ]:
# ── Cell 2: Config — edit these before running ──
import os

# ── Identity ──────────────────────────────────────────────────────────────────
RUN_NAME   = "gpt2-small-run001"
HF_USER    = "flakoash"
HF_TOKEN   = os.environ.get("HF_TOKEN")

HUB_CURRICULUM_REPO = None   # "flakoash/babylm-curriculum-run05"
HUB_MODEL_REPO      = f"{HF_USER}/babylm-{RUN_NAME}"

# ── Model architecture ────────────────────────────────────────────────────────
N_EMBD   = 384
N_LAYER  = 8
N_HEAD   = 6
N_INNER  = N_EMBD * 4
VOCAB_SIZE   = 16384
N_POSITIONS  = 1024

TOKENIZER_NAME = "BabyLM-community/BabyLM-2026-Baseline-GPT2-Strict-Small"

# ── Curriculum phases ─────────────────────────────────────────────────────────
# One entry per epoch_XX.jsonl in HUB_CURRICULUM_REPO.
# PHASE_EPOCHS controls the curriculum schedule (easy→hard).
# TOTAL_WORD_TARGET controls when training stops.
# If total scheduled words < TOTAL_WORD_TARGET, the last phase is extended
# automatically. Set to None to use PHASE_EPOCHS as-is.
PHASE_EPOCHS       = [5, 5]
TOTAL_WORD_TARGET  = 100_000_000   # 100M → all competition milestones reachable

# ── Training hyperparams ──────────────────────────────────────────────────────
MAX_SEQ_LEN      = 128
PER_DEVICE_BATCH = 64
EFFECTIVE_BATCH  = 256
LEARNING_RATE    = 7e-4
LR_SCHEDULER     = "cosine"
SEED             = 0

# ── Word-count milestones — full Strict-Small competition set ─────────────────
WORD_CHECKPOINTS = {
    **{i * 1_000_000:    f"chck_{i}M"     for i in range(1, 10)},   # 1M–9M
    **{i * 10_000_000:   f"chck_{i*10}M"  for i in range(1, 11)},   # 10M–100M
}

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR     = f"/workspace/outputs/{RUN_NAME}"
CURRICULUM_DIR = "/workspace/curriculum"

print(f"Run         : {RUN_NAME}")
print(f"Model repo  : {HUB_MODEL_REPO}")
print(f"Phases      : {PHASE_EPOCHS}  ({sum(PHASE_EPOCHS)} scheduled epochs)")
print(f"Word target : {TOTAL_WORD_TARGET:,}")
print(f"Milestones  : {list(WORD_CHECKPOINTS.values())}")

In [ ]:
# ── Cell 3: Imports + device ──
import json, os, re, sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel

for _p in [Path("../src"), Path("src"), Path("/workspace/babylm/src")]:
    if _p.exists() and str(_p.resolve()) not in sys.path:
        sys.path.insert(0, str(_p.resolve()))

from influence_curriculum.train import TrainingConfig, train_word_aware

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT    = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
Path(CURRICULUM_DIR).mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

In [ ]:
# ── Cell 4: Pull curriculum JSONL files from Hub ──
# Downloads epoch_00.jsonl, epoch_01.jsonl, ... skipping files already present.
#
# hf_hub_download preserves the repo's directory structure inside local_dir.
# Hub files are stored at  curriculum/epoch_NN.jsonl, so we pass the PARENT
# of CURRICULUM_DIR as local_dir to avoid the double-nesting
# (/workspace/curriculum/curriculum/...) that occurs otherwise.

phase_files = []

if HUB_CURRICULUM_REPO:
    from huggingface_hub import hf_hub_download, list_repo_files
    from pathlib import Path as _Path
    _curriculum_parent = str(_Path(CURRICULUM_DIR).parent)   # "/workspace"

    remote = sorted(
        f for f in list_repo_files(HUB_CURRICULUM_REPO, repo_type="dataset", token=HF_TOKEN)
        if f.startswith("curriculum/epoch_") and f.endswith(".jsonl")
    )
    print(f"Found {len(remote)} curriculum files on Hub.")
    for rpath in remote:
        local = Path(CURRICULUM_DIR) / Path(rpath).name   # /workspace/curriculum/epoch_NN.jsonl
        if not local.exists():
            hf_hub_download(HUB_CURRICULUM_REPO, rpath,
                            local_dir=_curriculum_parent,     # /workspace  (not /workspace/curriculum)
                            repo_type="dataset", token=HF_TOKEN)
            print(f"  downloaded {Path(rpath).name}")
        else:
            print(f"  {Path(rpath).name} already present")
        phase_files.append(str(local))
else:
    phase_files = sorted(str(p) for p in Path(CURRICULUM_DIR).glob("epoch_*.jsonl"))
    print(f"HUB_CURRICULUM_REPO not set — using local files: {[Path(p).name for p in phase_files]}")

assert len(phase_files) >= len(PHASE_EPOCHS), (
    f"Need {len(PHASE_EPOCHS)} JSONL files, found {len(phase_files)}"
)
phase_files = phase_files[:len(PHASE_EPOCHS)]
phases = list(zip(phase_files, PHASE_EPOCHS))

print("\nTraining phases:")
for i, (f, e) in enumerate(phases):
    n = sum(1 for _ in open(f))
    print(f"  Phase {i}: {Path(f).name}  {n:,} docs  {e} epoch(s)")

In [ ]:
# ── Cell 5: Resume check ──
start_phase, start_epoch, start_words = 0, 0, 0
model = GPT2LMHeadModel(GPT2Config(
    n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_inner=N_INNER,
    vocab_size=VOCAB_SIZE, n_positions=N_POSITIONS,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
))
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

if HUB_MODEL_REPO:
    from huggingface_hub import HfApi, list_repo_files
    api = HfApi()
    api.create_repo(repo_id=HUB_MODEL_REPO, repo_type="model",
                    exist_ok=True, token=HF_TOKEN)
    try:
        # Checkpoints are uploaded as phase_XX_epoch_YY/ at the repo root
        ckpt_pat = re.compile(r"^phase_(\d+)_epoch_(\d+)/config\.json$")
        remote_ckpts = [
            (int(m.group(1)), int(m.group(2)))
            for f in list_repo_files(HUB_MODEL_REPO, repo_type="model", token=HF_TOKEN)
            if (m := ckpt_pat.match(f))
        ]
        if remote_ckpts:
            from huggingface_hub import snapshot_download
            lp, le = max(remote_ckpts)
            ckpt_name = f"phase_{lp:02d}_epoch_{le:02d}"
            local_ckpt = OUT / "checkpoints" / ckpt_name
            if not local_ckpt.exists():
                print(f"Downloading {ckpt_name} from Hub...")
                snapshot_download(
                    repo_id=HUB_MODEL_REPO,
                    local_dir=str(OUT / "checkpoints"),
                    allow_patterns=f"{ckpt_name}/*",
                    local_dir_use_symlinks=False,
                    repo_type="model", token=HF_TOKEN,
                )
            model = GPT2LMHeadModel.from_pretrained(str(local_ckpt))

            # Recover words_processed
            hist_path = OUT / "training_history.json"
            if not hist_path.exists():
                try:
                    from huggingface_hub import hf_hub_download
                    hf_hub_download(HUB_MODEL_REPO, "training_history.json",
                                    local_dir=str(OUT), repo_type="model", token=HF_TOKEN)
                except Exception:
                    pass
            if hist_path.exists():
                hist = json.loads(hist_path.read_text())
                start_words = hist[-1].get("words_processed", 0)

            # Advance resume point past the completed epoch.
            # If lp is the last phase and its scheduled epochs are exhausted,
            # stay on it (train_word_aware will compute the extended epoch count).
            if lp + 1 < len(phases):
                if le + 1 < PHASE_EPOCHS[lp]:
                    start_phase, start_epoch = lp, le + 1
                else:
                    start_phase, start_epoch = lp + 1, 0
            else:
                # Last phase — may have extension epochs beyond PHASE_EPOCHS[-1]
                start_phase, start_epoch = lp, le + 1

            print(f"Resuming from {ckpt_name} → phase={start_phase} epoch={start_epoch} words={start_words:,}")
        else:
            print("No checkpoints on Hub — starting from scratch.")
    except Exception as e:
        print(f"Resume check failed ({e}) — starting from scratch.")
else:
    print("HUB_MODEL_REPO not set — no Hub push.")

In [ ]:
# ── Cell 6: Train ──

train_cfg = TrainingConfig(
    per_device_batch_size=PER_DEVICE_BATCH,
    effective_batch_size=EFFECTIVE_BATCH,
    learning_rate=LEARNING_RATE,
    lr_scheduler=LR_SCHEDULER,
    max_seq_len=MAX_SEQ_LEN,
    fp16=(DEVICE == "cuda"),
)

ckpt_paths, history = train_word_aware(
    model, tokenizer,
    phases=phases,
    output_dir=str(OUT),
    config=train_cfg,
    seed=SEED,
    device=DEVICE,
    word_checkpoints=sorted(WORD_CHECKPOINTS),
    checkpoint_names=WORD_CHECKPOINTS,
    hub_repo=HUB_MODEL_REPO,
    hub_token=HF_TOKEN,
    start_words=start_words,
    start_phase=start_phase,
    start_epoch=start_epoch,
    total_word_target=TOTAL_WORD_TARGET,
)

print(f"\nTraining complete — {len(ckpt_paths)} checkpoints saved.")

In [ ]:
# ── Cell 7: Push final model + metadata to Hub ──

# Identify the last per-epoch checkpoint as the final model
epoch_ckpts = [p for p in ckpt_paths if "phase_" in Path(p).name]
final_ckpt  = epoch_ckpts[-1] if epoch_ckpts else None

run_config = {
    "run_name":        RUN_NAME,
    "tokenizer":       TOKENIZER_NAME,
    "curriculum_repo": HUB_CURRICULUM_REPO,
    "phase_epochs":    PHASE_EPOCHS,
    "model": {
        "n_embd": N_EMBD, "n_layer": N_LAYER, "n_head": N_HEAD,
        "n_inner": N_INNER, "vocab_size": VOCAB_SIZE, "n_positions": N_POSITIONS,
    },
    "training": {
        "per_device_batch": PER_DEVICE_BATCH,
        "effective_batch":  EFFECTIVE_BATCH,
        "learning_rate":    LEARNING_RATE,
        "lr_scheduler":     LR_SCHEDULER,
        "max_seq_len":      MAX_SEQ_LEN,
        "seed":             SEED,
    },
    "word_checkpoints": WORD_CHECKPOINTS,
    "total_words_processed": history[-1]["words_processed"] if history else 0,
}

(OUT / "run_config.json").write_text(json.dumps(run_config, indent=2))
(OUT / "training_history.json").write_text(json.dumps(history, indent=2))
print("Metadata written.")

if HUB_MODEL_REPO and final_ckpt:
    from huggingface_hub import HfApi
    api = HfApi()

    # Final model weights → main branch root
    print(f"Pushing final model ({Path(final_ckpt).name}) → {HUB_MODEL_REPO} main ...")
    api.upload_folder(
        folder_path=final_ckpt,
        repo_id=HUB_MODEL_REPO,
        path_in_repo=".",
        repo_type="model", token=HF_TOKEN,
        commit_message="final model",
    )

    # Metadata files → main branch root
    for fname in ["run_config.json", "training_history.json"]:
        api.upload_file(
            path_or_fileobj=str(OUT / fname),
            path_in_repo=fname,
            repo_id=HUB_MODEL_REPO,
            repo_type="model", token=HF_TOKEN,
            commit_message=f"add {fname}",
        )
    print("Done.")
elif not final_ckpt:
    print("No per-epoch checkpoint found — skipping final model push.")

In [ ]:
# ── Cell 8: Training curves ──
import math
import matplotlib.pyplot as plt

# Load history (covers resume case where history may be partial)
hist_path = OUT / "training_history.json"
if hist_path.exists():
    history = json.loads(hist_path.read_text())

losses       = [h["loss"] for h in history]
perplexities = [math.exp(min(l, 20)) for l in losses]
lrs          = [h["lr"] for h in history]
words        = [h["words_processed"] for h in history]

# Phase boundaries (x-axis = epoch index)
phase_boundaries = []
running = 0
for ep in PHASE_EPOCHS[:-1]:
    running += ep
    phase_boundaries.append(running - 0.5)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
x = list(range(len(history)))

for ax, y, label, color in [
    (axes[0], losses,       "Loss",       "steelblue"),
    (axes[1], perplexities, "Perplexity", "darkorange"),
    (axes[2], lrs,          "LR",         "green"),
]:
    ax.plot(x, y, marker="o", color=color)
    for b in phase_boundaries:
        ax.axvline(b, color="red", linestyle="--", alpha=0.5, label="phase boundary" if ax is axes[0] else None)
    ax.set_ylabel(label)

# Annotate word-count milestones on loss plot
milestone_words = sorted(WORD_CHECKPOINTS)
for i, h in enumerate(history):
    for mw in milestone_words:
        if h["words_processed"] >= mw and (i == 0 or history[i-1]["words_processed"] < mw):
            axes[0].axvline(i, color="purple", linestyle=":", alpha=0.4)
            axes[0].text(i, max(losses) * 0.97, WORD_CHECKPOINTS[mw],
                         fontsize=7, color="purple", rotation=90, va="top")

axes[0].legend(fontsize=8)
axes[0].set_title(f"Training — {RUN_NAME}")
axes[2].set_xlabel("Epoch (global)")

plt.tight_layout()
plot_path = OUT / "training_curves.png"
plt.savefig(str(plot_path), dpi=150)
plt.show()
print(f"Saved to {plot_path}")

if HUB_MODEL_REPO:
    from huggingface_hub import HfApi
    HfApi().upload_file(
        path_or_fileobj=str(plot_path),
        path_in_repo="training_curves.png",
        repo_id=HUB_MODEL_REPO,
        repo_type="model", token=HF_TOKEN,
        commit_message="training curves",
    )
    print(f"Pushed training_curves.png → {HUB_MODEL_REPO}")